In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

Getting representative documents

In [ ]:
df = pd.read_csv("../../results/NLP_data_advice_fulltext.csv")
docs = df["text"].str.replace('\xa0', '', regex=False).tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    use_auth_token=False
)

embeddings = embedding_model.encode(docs, show_progress_bar=True)

topic_model = BERTopic.load(
    "../../results/NLP/BERTopic_model",
    embedding_model=embedding_model
)

doc_info = topic_model.get_document_info(docs)
topics = doc_info["Topic"].tolist()

train_idxs = np.arange(len(docs))

doc_topic = pd.DataFrame({
    "Topic": topics,
    "ID": train_idxs,
    "Document": [docs[i] for i in train_idxs]
})

topic_model._create_topic_vectors(doc_topic, embeddings[train_idxs])

repr_docs, _, _, _ = topic_model._extract_representative_docs(
    topic_model.c_tf_idf_,
    doc_topic,
    topic_model.topic_representations_,
    nr_samples=1000,
    nr_repr_docs=5
)

topic_model.representative_docs_ = repr_docs
topic_model.get_topic_info()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1241.64it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1169.91it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,133,-1_gnome_gnomes_blue_round,"[gnome, gnomes, blue, round, basket]","[colour gnome, gnomes, gnome, red basket, yell...","[colour gnome, yellow basket, gnome gives, num...","[There are 8 gnomes, each of varying colours. ..."
1,0,268,0_mushrooms_gnome_gnomes_try,"[mushrooms, gnome, gnomes, try, colour]","[gnomes mushrooms, coloured gnomes, number mus...","[gnomes mushrooms, coloured gnomes, number mus...",[Try each colour gnome at the beginning and if...
2,1,197,1_gnome_points_gnomes_hat,"[gnome, points, gnomes, hat, colour]","[colour gnomes, colour gnome, choose gnome, gn...","[colour gnomes, choose gnome, colours, points,...",[Be as quick as you can when choosing the gnom...
3,2,132,2_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes]","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, gnomes red basket, bask...","[There are 8 different color gnomes, split int..."
4,3,104,3_points_blue_pink_colours,"[points, blue, pink, colours, purple]","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, colours, colors...","[You'll see the colours in pairs, e.g. red and..."
5,4,82,4_basket_red_yellow_baskets,"[basket, red, yellow, baskets, points]","[basket colours, colour basket, basket colour,...","[basket colours, baskets red yellow, red baske...",[there will be 2 baskets (red and yellow) and ...
6,5,35,5_hats_tall_hat_tall hats,"[hats, tall, hat, tall hats, good]","[hat colour, tall hats, short hat, hats, small...","[hat colour, tall hats, smaller hat, yellow ha...",[The patterns appear to be inconsistent but I ...
7,6,27,6_keys_breaks_just_game,"[keys, breaks, just, game, fingers]","[press keys, stay focused, fingers keys, keybo...","[press keys, stay focused, fingers keys, make ...",[Have your fingers over the necessary keys at ...
8,7,22,7_forest_mushrooms_gnomes_green,"[forest, mushrooms, gnomes, green, orange]","[gnomes mushrooms, mushrooms gnomes, mushrooms...","[mushrooms gnomes, mushrooms forest, mushrooms...",[There are two distinct groups of gnomes(Group...


Save representative doc

In [24]:
topic_info = topic_model.get_topic_info()
rep_docs_expanded = pd.DataFrame(topic_info["Representative_Docs"].tolist())
out = pd.concat([topic_info[["Topic"]], rep_docs_expanded], axis=1)
out.to_csv("../../results/NLP/representative_docs.csv", index=False)

Save topic summary

In [25]:
topic_summary = topic_model.get_topic_info()
topic_summary.to_csv("../../results/NLP/topic_summary.csv")

Unclassified message

In [31]:
df = pd.read_csv("../../results/NLP_data_advice_fulltext.csv")
docs = df["text"].str.replace('\xa0', '', regex=False).tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    use_auth_token=False
)

topic_model = BERTopic.load(
    "../../results/NLP/BERTopic_model",
    embedding_model=embedding_model
)

doc_info = topic_model.get_document_info(docs)
topics = doc_info["Topic"].tolist()

new_topics = topic_model.reduce_outliers(docs, topics)
topic_model.update_topics(docs, topics=new_topics)
topic_model.get_topic_info()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1056.30it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 871.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:00<00:00, 24.13it/s]
2026-02-25 17:40:49,158 - BERTopic - WARNIN

,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,0,303,0_the_to_mushrooms_you,"[the, to, mushrooms, you, of, and, it, that, i...","[gnomes mushrooms, coloured gnomes, number mus...","[gnomes mushrooms, coloured gnomes, number mus...",NaN
1,1,237,1_the_to_gnome_and,"[the, to, gnome, and, gnomes, points, you, of,...","[colour gnomes, colour gnome, choose gnome, gn...","[colour gnomes, choose gnome, colours, points,...",NaN
2,2,145,2_basket_the_red_to,"[basket, the, red, to, yellow, you, and, mushr...","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, gnomes red basket, bask...",NaN
3,3,119,3_the_and_to_it,"[the, and, to, it, you, that, points, blue, fo...","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, colours, colors...",NaN
4,4,91,4_the_basket_to_and,"[the, basket, to, and, red, you, gnomes, yello...","[basket colours, colour basket, basket colour,...","[basket colours, baskets red yellow, red baske...",NaN
5,5,38,5_hats_the_and_to,"[hats, the, and, to, hat, tall, for, you, if, ...","[hat colour, tall hats, short hat, hats, small...","[hat colour, tall hats, smaller hat, yellow ha...",NaN
6,6,40,6_the_you_and_to,"[the, you, and, to, your, on, as, it, on the, ...","[press keys, stay focused, fingers keys, keybo...","[press keys, stay focused, fingers keys, make ...",NaN
7,7,27,7_forest_the_to_you,"[forest, the, to, you, mushrooms, other, the o...","[gnomes mushrooms, mushrooms gnomes, mushrooms...","[mushrooms gnomes, mushrooms forest, mushrooms...",NaN


save topic reduced dataframe

In [ ]:
topic_distr, _ = topic_model.approximate_distribution(docs)
df_stat = pd.read_csv("../../data/NLP_data_stake.csv")

for topic_n in range(len(topic_distr[0,:])):
    topic_name = "topic_" + str(topic_n)
    df_stat[topic_name] = topic_distr[:, topic_n]

df_stat["assigned_topic"] = topic_model.topics_

df_stat.head()
parent_topic_weight = []
var_names = [f'topic_{n}' for n in range(len(topic_distr[0, :]))]

for i in range(len(df_stat["ID"])):
    parent_ID = df_stat["parent_ID"][i]
    parent_row = df_stat[df_stat["ID"] == parent_ID]
    if len(parent_row) < 1:
        parent_topic_weight.append([None for n in range(len(topic_distr[0, :]))])
    else:
        res = parent_row[var_names].values.tolist()
        parent_topic_weight.append(res[0])

parent_topic_df = pd.DataFrame(parent_topic_weight)
parent_topic_df.columns = [f'parent_topic_{n}' for n in range(len(topic_distr[0, :]))]

df_stat = pd.concat([df_stat, parent_topic_df], axis=1)
df_stat = df_stat.loc[:, ~df_stat.columns.str.contains('^Unnamed')]

df_stat.to_csv("../../results/NLP/data_topic_weights_reduced.csv", header=True, index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.68it/s]


In [ ]:
df = pd.read_csv("../../results/NLP_data_advice_fulltext.csv")
docs = df["text"].str.replace('\xa0', '', regex=False).tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    use_auth_token=False
)

embeddings = embedding_model.encode(docs, show_progress_bar=True)

topic_model = BERTopic.load(
    "../../old/NLP/results/BERTopic_model",
    embedding_model=embedding_model
)

doc_info = topic_model.get_document_info(docs)
topics = doc_info["Topic"].tolist()

train_idxs = np.arange(len(docs))

doc_topic = pd.DataFrame({
    "Topic": topics,
    "ID": train_idxs,
    "Document": [docs[i] for i in train_idxs]
})

topic_model._create_topic_vectors(doc_topic, embeddings[train_idxs])

repr_docs, _, _, _ = topic_model._extract_representative_docs(
    topic_model.c_tf_idf_,
    doc_topic,
    topic_model.topic_representations_,
    nr_samples=1000,
    nr_repr_docs=5
)

topic_model.representative_docs_ = repr_docs
topic_model.get_topic_info()